# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [2]:
# importar librerías
import pandas as pd
import numpy as no
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from scipy.stats import levene
from scipy.stats import chi2_contingency

In [3]:
# cargar archivos
orders = pd.read_csv("rappiplus_orders_raw.csv") # tu código aquí
catalog = pd.read_csv("rappiplus_catalog.csv") # tu código aquí
marketing = pd.read_csv("rappiplus_marketing_spend.csv") # tu código aquí

In [4]:
# explorar datasets
# tu código aqui
def explorar_datasets(dataframes):
    resultados = [df.head() for df in dataframes]
    describe = [df.describe() for df in dataframes]
    info = [df.info() for df in dataframes]
    return resultados, describe, info

In [5]:
explorar_datasets([orders])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


([  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
  0   order_0  user_6993        2025-05-22  Argentina     desktop   
  1   order_1  user_1329        2025-06-15     Mexico     desktop   
  2   order_2  user_3194        2025-05-02  Argentina     desktop   
  3   order_3  user_4510        2025-06-09   Colombia      mobile   
  4   order_4  user_5044        2025-03-30  Argentina     desktop   
  
    fuente_referencia       nombre_producto categoria_producto  cantidad  \
  0           organic       Jacket-Winter-M               Moda       2.0   
  1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
  2            social        Blender-XL-Red              Hogar       2.0   
  3            social  Tablet-Standard-64GB        Electronica       1.0   
  4       paid_search        Blender-XL-Red              Hogar       1.0   
  
     precio_unitario  monto_descuento  monto_total  
  0           332.69              0.0       665.37  
  1           176.

In [6]:
explorar_datasets([catalog])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


([        nombre_producto categoria_producto  costo_unitario  \
  0    Laptop-Gaming-16GB        Electrónica          280.68   
  1       Phone-Pro-128GB        Electrónica           10.12   
  2  Tablet-Standard-64GB        Electrónica           25.21   
  3        Blender-XL-Red              Hogar          176.64   
  4      Vacuum-Pro-Black              Hogar           16.60   
  
                   proveedor  
  0   Fuller, Pena and Myers  
  1                 King Ltd  
  2               Bowers LLC  
  3                Long-Reid  
  4  Rivera, Carr and Finley  ],
 [       costo_unitario
  count        7.000000
  mean       102.252857
  std        111.011563
  min         10.120000
  25%         16.905000
  50%         25.210000
  75%        182.975000
  max        280.680000],
 [None])

In [7]:
explorar_datasets([marketing])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


([        fecha      pais            id_campaña        canal    gasto
  0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
  1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
  2  2025-01-01    Mexico         social_Mexico       social  2045.01
  3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
  4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40],
 [            gasto
  count  1620.00000
  mean   1772.74292
  std     734.43294
  min     501.11000
  25%    1128.03000
  50%    1782.42500
  75%    2420.68500
  max    2999.36000],
 [None])

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [8]:
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"])
marketing["fecha"] = pd.to_datetime(marketing["fecha"])

In [9]:
#Empezamos con las filas duplicadas hay que irlas quitando

orders = orders.drop_duplicates(keep="first")

In [10]:
#Comprobamos el cambio realizado
orders.duplicated().sum()

0

Ahora ya que nos deshicimos de los duplicados, falta atender todos los nulos para que no afecte a nuestros analisis.

**Hasta ahora contamos con nulos en:**
- Pais (300)
- Dispositivo (20)
- Fuente referencia (30)
- Nombre_producto (30)
- Categoria_producto (80)
- Cantidad (50)
- Precio_unitario (50)
- Monto_descuento (50)

In [11]:
#En este caso optamos por imputar nulos con la moda
orders["dispositivo"] = orders["dispositivo"].fillna("desktop")

In [12]:
#Vamos con País
orders["pais"].unique()

array(['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia',
       'argentina', nan], dtype=object)

In [13]:
#Esas faltas ortograficas solo entorpeceran nuestro analisis asi que debemos corregir los nombres de paises mal escritos:
orders["pais"] = orders["pais"].replace({
    "mexico": "Mexico",
    "colombia": "Colombia",
    "argentina": "Argentina"
})
orders["pais"].unique()

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

In [14]:
#Las siguientes columnas con variable categorica he optado por imputar nulos con "otro" para no eliminar filas ademas de que no hay una explicaicón clara de porque estos nulos.
orders[["pais", "fuente_referencia", "nombre_producto", "categoria_producto"]] = orders[["pais", "fuente_referencia", "nombre_producto", "categoria_producto"]].fillna("Otro")
orders[["pais", "fuente_referencia", "nombre_producto", "categoria_producto"]].isna().sum()

pais                  0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
dtype: int64

In [15]:
#Le toca las variables númericas, vemos que tambien son muy pocas, asi que sin problema podemos imputar con la mediana
columns = ["cantidad", "precio_unitario", "monto_descuento"]
for col in columns:
    orders[col] = orders[col].fillna(orders[col].median())

In [16]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                25000 non-null  object        
 4   dispositivo         25000 non-null  object        
 5   fuente_referencia   25000 non-null  object        
 6   nombre_producto     25000 non-null  object        
 7   categoria_producto  25000 non-null  object        
 8   cantidad            25000 non-null  float64       
 9   precio_unitario     25000 non-null  float64       
 10  monto_descuento     25000 non-null  float64       
 11  monto_total         25000 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [17]:
#Todo listo! Ya no tenemos nulos en ninguna columna, pero aun hay un problema en cantidad y monto total
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25000.000000,25000.000000,25000.000000,2.500000e+04
mean,7.104880,259.373368,4.493600,2.079498e+03
std,296.573019,138.551974,5.222705,9.914760e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.705000,0.000000,1.806700e+02
50%,2.000000,258.795000,0.000000,3.418050e+02
75%,2.000000,380.122500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [18]:
orders.sort_values(by="cantidad", ascending=False).head(20)
#Muy sospechoso los valores pasan de maximo 2 unidades a 20000


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.00
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.00
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.00
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.00
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.00
3689,order_3689,user_6566,2025-06-16,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.00
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.00
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.00
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.00
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.00


In [19]:
#Como la columna monto_total esta contaminada por los negativos y los sentinels que ya encontramos hay que eliminarla y volverla a hacer?
orders = orders.drop(columns=["monto_total"])

In [20]:
#Confirmamos viendo los valores unicos que esos sentinels se puedan deber a un error
orders["cantidad"].value_counts().sort_index()

-2.0            1
-1.0            3
 1.0        12345
 2.0        12641
 10000.0        6
 20000.0        4
Name: cantidad, dtype: int64

En efecto, todo dice que no deberian estar ahi.

In [21]:
#Con lo que ya vimos ponemos deducir que los outliers se deben a un error de sistema o de captura, asi que hay que reemplazarlos por 2 y 1
orders["cantidad"] = orders["cantidad"].replace({
    20000: 2.0,
    10000: 1.0

})


In [22]:

orders["cantidad"] = orders["cantidad"].replace({
    -2.0: 2.0,
    -1.0: 1.0
})

In [23]:
#Ahora ponemos rehacemos la columna de monto total como "total" y ya tenemos limpio el dataframe
orders["total"] = (orders["precio_unitario"] * orders["cantidad"]) - orders["monto_descuento"]
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [24]:
#Solo queda ahora Marketing en el canal de ventas donde tenemos 101 nulos
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


Aqui todo es mucho mas sencillo, hay un fallo en el sistema y para arreglar esto, solo tenemos que fijarnos en id_campañama y ahi nos dice el canal (Ejemplo: si es organic_mexico el canal seria organic)

In [25]:
marketing["canal"] = marketing["canal"].fillna(
    marketing["id_campaña"].str.rsplit("_", n=1).str[0]
)

In [26]:
#Comprobamos ahora
marketing.isna().sum()

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

Listo!! Nuestros Dataframes ya estan limpios y listos para el analisis

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [27]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [28]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [29]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [30]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [31]:
cantidad_producto = (orders.groupby("nombre_producto")["cantidad"].sum().reset_index())

In [32]:
catalog = catalog.merge(cantidad_producto,on="nombre_producto", how="inner")
catalog

,nombre_producto,categoria_producto,costo_unitario,proveedor,cantidad
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers",4220.0
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd,4151.0
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC,4163.0
3,Blender-XL-Red,Hogar,176.64,Long-Reid,6285.0
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley",6296.0
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith,6210.0
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes,6276.0


In [33]:
catalog["costo_total"] = catalog["costo_unitario"] * catalog["cantidad"]
catalog

,nombre_producto,categoria_producto,costo_unitario,proveedor,cantidad,costo_total
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers",4220.0,1184469.60
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd,4151.0,42008.12
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC,4163.0,104949.23
3,Blender-XL-Red,Hogar,176.64,Long-Reid,6285.0,1110182.40
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley",6296.0,104513.60
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith,6210.0,106874.10
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes,6276.0,1188109.56


In [34]:
orders_profit = orders.merge(catalog[["nombre_producto", "costo_unitario"]], on="nombre_producto",how="left")

orders_profit["costo_total"] = (orders_profit["cantidad"] * orders_profit["costo_unitario"])

orders_profit["profit"] = (orders_profit["total"] - orders_profit["costo_total"])

orders_profit.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,total,costo_unitario,costo_total,profit
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38,189.31,378.62,286.76
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,25.21,146.65
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98,176.64,353.28,-157.30
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,25.21,217.66
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,176.64,159.64


In [35]:
ingreso_total = round(orders["total"].sum(), 2)
costo_total_final = round(catalog["costo_total"].sum(), 2)
inversion_total_marketing = marketing["gasto"].sum()
profit_total = round(orders_profit["total"].sum() - catalog["costo_total"].sum(), 2)
ticket_promedio = round(orders["total"].mean(), 2)
cantidad_promedio = round(orders["cantidad"].mean(), 0)
producto_mas_vendido = (orders.groupby("nombre_producto")["cantidad"].sum().idxmax())
producto_mas_rentable = (orders_profit.groupby("nombre_producto")["profit"].sum().idxmax())

In [36]:
print("KPIs:")
print()
print("Ingreso Total :", ingreso_total)
print()
print("Costo Total :", costo_total_final)
print()
print("Costo Total :", profit_total)
print()
print("Mejor profit :", producto_mas_rentable)
print()
print("Total Invertido en Marketing :", inversion_total_marketing)
print()
print("Ticket Promedio :", ticket_promedio)
print()
print("Cantidad Promedio :", cantidad_promedio)
print()
print("Producto mas vendido :", producto_mas_vendido)

KPIs:

Ingreso Total : 9653543.64

Costo Total : 3841106.61

Costo Total : 5812437.03

Mejor profit : Vacuum-Pro-Black

Total Invertido en Marketing : 2871843.53

Ticket Promedio : 386.14

Cantidad Promedio : 2.0

Producto mas vendido : Vacuum-Pro-Black


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [37]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [38]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [39]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id_usuario          120000 non-null  object
 1   id_sesion           120000 non-null  object
 2   nombre_evento       120000 non-null  object
 3   timestamp_evento    120000 non-null  object
 4   pais                120000 non-null  object
 5   dispositivo         120000 non-null  object
 6   fuente_referencia   120000 non-null  object
 7   categoria_producto  120000 non-null  object
dtypes: object(8)
memory usage: 7.3+ MB


In [40]:
events.describe()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
count,120000,120000,120000,120000,120000,120000,120000,120000
unique,8000,120000,6,181,3,2,3,3
top,user_159,e4500722-c4d5-4d41-9cd7-327d60fcbe9c,first_visit,2025-06-11,Colombia,desktop,paid_search,Electronica
freq,33,1,29957,741,40490,61038,40134,40125


In [41]:
events.dtypes

id_usuario            object
id_sesion             object
nombre_evento         object
timestamp_evento      object
pais                  object
dispositivo           object
fuente_referencia     object
categoria_producto    object
dtype: object

In [42]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE
        WHEN nombre_evento = 'first_visit' THEN 1
        WHEN nombre_evento = 'add_to_cart' THEN 2
        WHEN nombre_evento = 'add_payment_info' THEN 3
        WHEN nombre_evento = 'purchase' THEN 4
        ELSE 5
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,add_payment_info,6250
3,purchase,6240
4,begin_checkout,7208
5,select_item,7582


In [43]:

# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE 
            WHEN nombre_evento = 'first_visit' 
            THEN id_usuario 
        END) AS first_visit,

        COUNT(DISTINCT CASE 
            WHEN nombre_evento = 'add_to_cart' 
            THEN id_usuario 
        END) AS add_to_cart,

        COUNT(DISTINCT CASE 
            WHEN nombre_evento = 'add_payment_info' 
            THEN id_usuario 
        END) AS add_payment_info,

        COUNT(DISTINCT CASE 
            WHEN nombre_evento = 'purchase' 
            THEN id_usuario 
        END) AS purchase

    FROM events
)

SELECT
    ROUND(add_to_cart * 100.0 / first_visit, 2) 
        AS conversion_visit_cart,

    ROUND(add_payment_info * 100.0 / add_to_cart, 2) 
        AS conversion_cart_payment,

    ROUND(purchase * 100.0 / add_payment_info, 2) 
        AS conversion_payment_purchase,

    ROUND(purchase * 100.0 / first_visit, 2) 
        AS conversion_total
FROM funnel; '''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversion_visit_cart,conversion_cart_payment,conversion_payment_purchase,conversion_total
0,97.92,81.87,99.84,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [44]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [45]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [46]:
query_cohort_retention_final = """

WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC(
            'month',
            CAST(fecha_registro AS DATE)
        ) AS cohorte
    FROM users
),

actividad AS (
    SELECT
        c.id_usuario,
        c.cohorte,

        FLOOR(
            (
                CAST(a.fecha_actividad AS DATE)
                - c.fecha_registro
            ) / 7.0
        ) AS semana

    FROM cohortes c

    JOIN user_activity a
        ON c.id_usuario = a.id_usuario
       AND a.activo = 1
),

actividad_usuario AS (
    SELECT
        cohorte,
        id_usuario,

        MAX(
            CASE
                WHEN semana = 1 THEN 1
                ELSE 0
            END
        ) AS activo_w1,

        MAX(
            CASE
                WHEN semana = 2 THEN 1
                ELSE 0
            END
        ) AS activo_w2,

        MAX(
            CASE
                WHEN semana = 3 THEN 1
                ELSE 0
            END
        ) AS activo_w3

    FROM actividad

    GROUP BY
        cohorte,
        id_usuario
),

retencion AS (
    SELECT
        cohorte,

        COUNT(
            CASE
                WHEN activo_w1 = 1
                THEN 1
            END
        ) AS retenido_w1,

        COUNT(
            CASE
                WHEN activo_w1 = 1
                 AND activo_w2 = 1
                THEN 1
            END
        ) AS retenido_w2,

        COUNT(
            CASE
                WHEN activo_w1 = 1
                 AND activo_w2 = 1
                 AND activo_w3 = 1
                THEN 1
            END
        ) AS retenido_w3

    FROM actividad_usuario

    GROUP BY cohorte
),

usuarios_cohorte AS (
    SELECT
        cohorte,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales

    FROM cohortes

    GROUP BY cohorte
)

SELECT
    u.cohorte,
    u.usuarios_iniciales,

    COALESCE(r.retenido_w1, 0) AS retenido_w1,
    COALESCE(r.retenido_w2, 0) AS retenido_w2,
    COALESCE(r.retenido_w3, 0) AS retenido_w3,

    ROUND(
        100.0 * COALESCE(r.retenido_w1, 0)
        / u.usuarios_iniciales,
        2
    ) AS semana_1,

    ROUND(
        100.0 * COALESCE(r.retenido_w2, 0)
        / u.usuarios_iniciales,
        2
    ) AS semana_2,

    ROUND(
        100.0 * COALESCE(r.retenido_w3, 0)
        / u.usuarios_iniciales,
        2
    ) AS semana_3

FROM usuarios_cohorte u

LEFT JOIN retencion r
    ON u.cohorte = r.cohorte

ORDER BY u.cohorte;

"""

cohorte_final = pd.read_sql(
    query_cohort_retention_final,
    con=engine
)

cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,340,146,42.84,20.90,8.97
1,2025-02-01 00:00:00+00:00,1444,611,267,129,42.31,18.49,8.93
2,2025-03-01 00:00:00+00:00,1636,677,300,133,41.38,18.34,8.13
3,2025-04-01 00:00:00+00:00,1606,680,314,146,42.34,19.55,9.09
4,2025-05-01 00:00:00+00:00,1687,695,299,151,41.20,17.72,8.95


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión es igual entre la variante control y la variante tratamiento.
   - **H₁ (Hipótesis alternativa):** La tasa de conversión es diferente entre la variante control y la variante tratamiento.
   
- **Test estadístico:** Chi cuadrada sobre tabla de contingencia de 2x2 
- **Nivel de significancia alpha:** 0.05

In [47]:
# tu código aquí
df_experiment = pd.read_csv("experiment_checkout_ui.csv")
df_experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [48]:
df_experiment[["variante", "convirtio"]].value_counts()

variante     convirtio
tratamiento  0            4215
control      0            4186
tratamiento  1             820
control      1             779
dtype: int64

In [49]:
tabla = pd.crosstab( df_experiment["variante"], df_experiment["convirtio"])
tabla

convirtio,0,1
variante,,
control,4186,779
tratamiento,4215,820


In [50]:
tabla_norm = pd.crosstab( df_experiment["variante"], df_experiment["convirtio"], normalize='index') * 100
tabla_norm 

convirtio,0,1
variante,,
control,84.310171,15.689829
tratamiento,83.714002,16.285998


In [51]:
chi2_contingency(tabla)

Chi2ContingencyResult(statistic=0.6177857659309294, pvalue=0.43187130118526496, dof=1, expected_freq=array([[4171.0965,  793.9035],
       [4229.9035,  805.0965]]))

In [52]:
chi2_stat, p_value, dof, expected = chi2_contingency(tabla)

In [53]:
print(f"Estadistico chi-cuadrado: {chi2_stat: 3f}")
print(f"valor P: {p_value:.3f}")
print(f"Grados de libertad: {dof}")
print("\nFrecuencias esperadas:")
print(expected)

Estadistico chi-cuadrado:  0.617786
valor P: 0.432
Grados de libertad: 1

Frecuencias esperadas:
[[4171.0965  793.9035]
 [4229.9035  805.0965]]


In [54]:
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipotesis nula: existe evidencia estadísticamente significativa de que la tasa de conversión difiere entre la variante control y tratamiento.")
else: 
    print("No rechazamos la hipotes nula: no existe evidencia estadísticamente significativa de una diferencia en la tasa de conversión entre la variante control y tratamiento.")

No rechazamos la hipotes nula: no existe evidencia estadísticamente significativa de una diferencia en la tasa de conversión entre la variante control y tratamiento.


In [55]:
orders

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
...,...,...,...,...,...,...,...,...,...,...,...,...
24995,order_24995,user_4217,2025-04-16,Argentina,mobile,social,Tablet-Standard-64GB,Electronica,1.0,64.57,0.0,64.57
24996,order_24996,user_3945,2025-02-14,Argentina,mobile,social,Vacuum-Pro-Black,Hogar,2.0,292.90,15.0,570.80
24997,order_24997,user_3119,2025-03-27,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,226.85,0.0,226.85
24998,order_24998,user_7218,2025-02-01,Mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,418.06,0.0,836.12


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
https://drive.google.com/drive/folders/1GamNd5mokv4PPZCJAhEHFY_FgIbuNRj4?usp=sharing

Mis Dashboards de Power BI se podran encontrar en el enlace de google drive o los archivos de este notebook ya que mi cuenta de universidad de Power BI muestra problemas para compartirse.